# Running and Analyzing Experiments

This notebook defines a repeatable experiment loop for baseline and constrained conditions across multiple random seeds.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root


In [ ]:
import numpy as np
import pandas as pd

from experiments.core import Distinction, DistinctionSet
from experiments.models.baseline import BaselineComplexSystemModel
from experiments.models import run_simulation


In [ ]:
distinction = Distinction(
    name="feature-0-invariant",
    parameters={
        "feature_index": 0,
        "target_value": 0.5,
        "tolerance": 0.15,
        "min_fraction": 0.6,
    },
)

def run_trial(seed: int, with_distinction: bool, steps: int = 100) -> dict:
    model = BaselineComplexSystemModel(
        n_agents=20,
        n_features=3,
        interaction_strength=0.3,
        seed=seed,
    )
    distinction_set = DistinctionSet([distinction]) if with_distinction else DistinctionSet([])
    result = run_simulation(model, steps=steps, distinctions=distinction_set)

    trajectory = np.array([state.state for state in result.trajectory])
    divergence = float(np.linalg.norm(trajectory[-1] - trajectory[0]))
    variability = float(np.var(trajectory))

    feature = trajectory[:, distinction.parameters["feature_index"]::3]
    compliance = float((np.abs(feature - distinction.parameters["target_value"]) <= distinction.parameters["tolerance"]).mean())

    return {
        "seed": seed,
        "condition": "distinction" if with_distinction else "baseline",
        "divergence": divergence,
        "variability": variability,
        "compliance": compliance,
    }


In [ ]:
seeds = list(range(10, 20))
rows = []

for seed in seeds:
    rows.append(run_trial(seed, with_distinction=False))
    rows.append(run_trial(seed, with_distinction=True))

df = pd.DataFrame(rows)
df.head()


In [ ]:
summary = (
    df.groupby("condition")
    .agg(
        divergence_mean=("divergence", "mean"),
        divergence_std=("divergence", "std"),
        variability_mean=("variability", "mean"),
        compliance_mean=("compliance", "mean"),
    )
    .reset_index()
)

summary


In [ ]:
output_path = repo_root / "results" / "notebook_experiments.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

output_path
